# Lab 2 — When history outgrows the window (EP07)

Lab 1: let the API cut for us and important messages silently disappear → this time **the harness chooses what to keep**

The same conversation, the same history budget, four methods from §18.2.3–18.2.4:
1. **FIFO** — drop the oldest messages first
2. **Pinned** — pin the user's first message, FIFO for the rest
3. **Importance-weighted** — weight each message, then keep them knapsack-style (Equation 18.6)
4. **Summarization** — summarize the old part into one paragraph, keep the recent part word for word (Equation 18.4)

Then code scores each method: is the answer in Thai · does it include papers older than 2023, which the user told us to leave out · does it have all the right papers

The paper notes in the conversation are **simulated data written for teaching** (the titles are real papers). Every model answer and summary comes from a real run.

> Source: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §18.2.3–18.2.4 — CC BY-SA 4.0

## 0. Setup

| You need | Details |
|---|---|
| Python 3.10 or later | or Google Colab, which already has it |
| An OpenRouter API key | sign up at openrouter.ai → Keys → Create Key, then add credit (this lab costs under one US cent) |
| Packages | `openai`, `python-dotenv`, `tokenizers` (installed in the next cell) |

**Adding the API key**
- **Colab:** the 🔑 Secrets icon → name `OPENROUTER_API_KEY` → turn on *Notebook access*
- **Your own machine:** create a `.env` file next to the notebook with the line `OPENROUTER_API_KEY=sk-or-...`

⚠️ Never put the key in the code, and never share your `.env` file

In [ ]:
%pip install -q "openai>=1.50" python-dotenv "tokenizers==0.23.3"

## 1. The model and token counting (from Lab 1)

Gemma 4 26B through OpenRouter, provider pinned to DeepInfra — why we chose the model and how to set up the key are in Lab 1
`message_tokens` counts one message's tokens in Gemma's chat format (within 1 token of the API)

In [ ]:
import json, os, re
from openai import OpenAI
os.environ["HF_HUB_VERBOSITY"] = "error"
from tokenizers import Tokenizer

def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key())
MODEL = "google/gemma-4-26b-a4b-it"
EXTRA = {"reasoning": {"enabled": False}, "provider": {"order": ["deepinfra/fp8"], "allow_fallbacks": False}}
USAGE = {"in": 0, "out": 0}

def chat(messages, max_tokens=400, **kw):
    reply = client.chat.completions.create(model=MODEL, messages=messages, max_tokens=max_tokens, extra_body=EXTRA | kw)
    USAGE["in"] += reply.usage.prompt_tokens
    USAGE["out"] += reply.usage.completion_tokens
    return reply.choices[0].message.content

tok = Tokenizer.from_pretrained(MODEL)

def count_tokens(text):
    return len(tok.encode(text, add_special_tokens=False).ids)

def message_tokens(m):
    role = "model" if m["role"] == "assistant" else m["role"]
    return count_tokens(f"<|turn>{role}\n{m['content'].strip()}<turn|>\n")

## 2. A conversation over budget

The user talks with the research assistant (simulated data):
- first message: **the task** — writing a review on transformers for time series · **always answer in Thai**
- third message: **a correction** — only papers from 2023 onward
- then: notes on 10 papers (2019–2024), mixed with small talk

This lab's history budget: **400 tokens** — kept small so the effect shows quickly and cheaply
(for real, Equation 18.2 gives H ≤ 50% of C ≈ 131,000 tokens, but the principle is the same)

In [ ]:
PAPERS = {  # simulated notes written for teaching; the titles and years are real papers
    "P01": ("Informer", 2021, "ProbSparse self-attention cuts the cost of long inputs; tested on electricity and weather data."),
    "P02": ("Autoformer", 2021, "Replaces attention with auto-correlation and splits each series into trend and seasonal parts."),
    "P03": ("FEDformer", 2022, "Works in the frequency domain with Fourier and wavelet blocks; linear cost in sequence length."),
    "P04": ("Are Transformers Effective for Time Series Forecasting?", 2023, "A one-layer linear model beats several transformers."),
    "P05": ("PatchTST", 2023, "Cuts each series into patches of 16 points and treats channels independently."),
    "P06": ("iTransformer", 2024, "Inverts the axes: each variable becomes a token, attention runs across variables."),
    "P07": ("TimesFM", 2024, "A decoder-only foundation model pretrained on 100 billion time points; good zero-shot forecasts."),
    "P08": ("Chronos", 2024, "Scales and quantizes values into tokens and trains a T5 language model on them."),
    "P09": ("Temporal Fusion Transformer", 2019, "LSTM encoders with attention and variable selection for interpretable forecasts."),
    "P10": ("Moirai", 2024, "A universal forecasting transformer trained on 27 billion observations across nine domains."),
}
SMALL_TALK = ["Thanks!", "Nice, coffee break, back in five minutes.", "Haha, my cat just walked on the keyboard.", "Ok, next one."]

HISTORY = [
    {"role": "user", "content": "Task for this whole chat: I'm writing a literature review on transformers for time series. I'll paste paper notes. Always answer in Thai."},
    {"role": "assistant", "content": "เข้าใจแล้วครับ ส่งโน้ตของ paper มาได้เลย"},
    {"role": "user", "content": "Correction to the task: my review only covers papers published in 2023 or later. Leave older ones out."},
    {"role": "assistant", "content": "รับทราบครับ จะใช้เฉพาะ paper ตั้งแต่ปี 2023"},
]
for i, pid in enumerate(["P04", "P05", "P06", "P01", "P07", "P09", "P08", "P02", "P10", "P03"]):
    title, year, note = PAPERS[pid]
    HISTORY += [{"role": "user", "content": f"Paper note {pid}: {title} ({year}). {note}"},
                {"role": "assistant", "content": f"บันทึก {pid} แล้วครับ"}]
    if i % 3 == 2:
        HISTORY += [{"role": "user", "content": SMALL_TALK[i // 3]}, {"role": "assistant", "content": "ได้เลยครับ"}]

In [ ]:
def label(m):
    t = m["content"]
    if t.startswith("Task"): return "T"
    if t.startswith("Correction"): return "C"
    if t.startswith("Paper note"): return t[11:14]
    return "a" if m["role"] == "assistant" else "s"

def show(kept, history=HISTORY):
    ids = {id(m) for m in kept}
    print(" ".join(label(m) if id(m) in ids else "·" for m in history))
    print(f"kept {len(kept)}/{len(history)} messages, {sum(message_tokens(m) for m in kept)} tokens")

In [ ]:
BUDGET = 400
show(HISTORY)

In [ ]:
SYSTEM = {"role": "system", "content": "You are the Cogito Learn research assistant."}
ASK = "List every paper that belongs in my review, one line each: id, title, year."
RESULTS = {}

def ask(name, kept):
    answer = chat([SYSTEM] + kept + [{"role": "user", "content": ASK}])
    ids = sorted(set(re.findall(r"^\W*(P\d\d)", answer, re.M)))  # ids that start a line = the listed papers
    RESULTS[name] = {"thai": sum(0x0E00 <= ord(ch) <= 0x0E7F for ch in answer) > 20,
                     "older than 2023": [p for p in ids if PAPERS[p][1] < 2023],
                     "right papers": f"{sum(PAPERS[p][1] >= 2023 for p in ids)}/6"}
    print(answer, "\n→", RESULTS[name])

In [ ]:
ask("full history", HISTORY)

## 3. FIFO — drop the oldest messages first

Keep messages from the newest backwards until the budget is full

In [ ]:
def fifo(history, budget):
    kept, used = [], 0
    for m in reversed(history):
        if used + message_tokens(m) > budget:
            break
        kept.insert(0, m)
        used += message_tokens(m)
    return kept

kept = fifo(HISTORY, BUDGET)
show(kept)
ask("FIFO", kept)

## 4. Pinned — pin the first message

The textbook: *"Keep the system prompt and first user message pinned"* — our system prompt is already outside the history,
so we pin the user's first message (the task) and run FIFO on the rest with what's left of the budget

In [ ]:
def pinned(history, budget):
    first = history[0]
    return [first] + fifo(history[1:], budget - message_tokens(first))

kept = pinned(HISTORY, BUDGET)
show(kept)
ask("pinned", kept)

## 5. Importance-weighted — keep messages knapsack-style (Equation 18.6)

Give each message a weight wᵢ (the textbook: tool results and user corrections weigh more), then keep the messages with the highest **wᵢ / |mᵢ|** first
— like packing a weight-limited suitcase: what's most worth its weight goes in first

| Message | w (our choice) |
|---|---|
| task / correction (T, C) | 10 |
| paper note | 4 |
| agent acknowledgement | 0.5 |
| small talk | 0.1 |

In [ ]:
def weight(m):
    t = m["content"]
    if m["role"] == "user" and t.startswith(("Task", "Correction")): return 10
    if t.startswith("Paper note"): return 4
    if m["role"] == "assistant": return 0.5
    return 0.1

def importance(history, budget):
    order = sorted(range(len(history)), key=lambda i: weight(history[i]) / message_tokens(history[i]), reverse=True)
    keep, used = set(), 0
    for i in order:
        if used + message_tokens(history[i]) <= budget:
            keep.add(i)
            used += message_tokens(history[i])
    return [history[i] for i in sorted(keep)]

kept = importance(HISTORY, BUDGET)
show(kept)
ask("importance", kept)

## 6. Summarization — summarize the old part (Equation 18.4)

**H′ = Summarize(H₁:ₖ) ‖ Hₖ₊₁:ₙ** — summarize the old messages into one paragraph, then append the recent messages word for word

The summarizer's prompt says to **keep every instruction and correction from the user, and every paper with its year**, and add nothing (temperature 0)
The textbook notes a smaller, cheaper model can do the summarizing — this lab uses the same Gemma

In [ ]:
SUMMARY_PROMPT = ("Summarize this conversation for the assistant's own memory, in English, in at most 120 words. "
                  "Keep every instruction and correction the user gave, and every paper id with its title and year. "
                  "Copy numbers exactly; add nothing.")

def summarize_old(history, keep_last=6):
    old, recent = history[:-keep_last], history[-keep_last:]
    text = "\n".join(f"{m['role']}: {m['content']}" for m in old)
    summary = chat([{"role": "user", "content": SUMMARY_PROMPT + "\n\n" + text}], max_tokens=300, temperature=0)
    print(f"summary: {sum(message_tokens(m) for m in old)} tokens → {count_tokens(summary)} tokens\n{summary}\n")
    return [{"role": "user", "content": "Summary of our earlier conversation: " + summary},
            {"role": "assistant", "content": "รับทราบครับ"}] + recent

kept = summarize_old(HISTORY)
print(sum(message_tokens(m) for m in kept), "tokens, budget", BUDGET)
ask("summary", kept)

In [ ]:
for name, r in RESULTS.items():
    print(f"{name:13} thai={str(r['thai']):5}  older than 2023={str(r['older than 2023']):28} right papers={r['right papers']}")
cost = USAGE["in"] / 1e6 * 0.07 + USAGE["out"] / 1e6 * 0.34
print(f"\nthe whole lab used {USAGE['in']:,} input + {USAGE['out']:,} output tokens ≈ ${cost:.4f}")

## Summary

| Method | What we saw in the lab |
|---|---|
| FIFO | The task and the correction are gone → older papers the user excluded show up (sometimes the answer is in English) |
| Pinned | Answers in Thai (the task is pinned) but the correction is gone → older papers still show up |
| Importance-weighted | Every instruction kept, no older paper, but one long note didn't fit → 5/6 |
| Summarization | The old part over 3× shorter · 6/6, like the full history · at the price of one extra LLM call |

**Exercises**
1. **Hierarchical summarization** (§18.2.4): split the old messages into two spans — the oldest summarized into one sentence, the middle into a paragraph, the recent kept word for word. Does it fit a 300-token budget, and is it still 6/6?
2. In `weight`, give small talk 4, the same as a paper note, and run `importance` again — what gets dropped instead?

**Next — Lab 3:** when a document is bigger than the budget from the start — split it into chunks, call the model on each chunk, then combine the results (recursive context decomposition)